# Лабораторна 3. Токенізація та трансформери

У цій лабораторній роботі ви пройдете шлях від механіки токенізації до практичного використання готових мовних моделей:

- алгоритми токенізації (BPE, WordPiece, Unigram) — натренуєте всі три на власному тексті;
- mini-GPT з нуля — повторите тренування трансформера на своєму датасеті;
- готові моделі для різних NLP-задач — класифікація, генерація, переклад, Q&A та інше через `pipeline()`;
- порівняння кількох моделей на одній задачі;
- творчий застосунок на основі готової моделі.

Демо-ноутбук лекції 4 (`transformers_demo.ipynb`) — основа для секцій 2-3, звертайтесь до нього як до довідника. Він лежить в архіві демо-матеріалів лекції 4 — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу. Розділи 1 і 4 спираються на офіційний [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter1/1) — читайте відповідні розділи курсу перед виконанням, посилання додані в кожній секції.

## Матеріали

- [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)
- [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [3Blue1Brown — How might LLMs store facts](https://www.youtube.com/watch?v=9-Jl0dxWQs8)
- [Hugging Face LLM Course — Chapter 1: Transformer Models](https://huggingface.co/learn/llm-course/en/chapter1/1)
- [Hugging Face LLM Course — Chapter 2: Using Transformers](https://huggingface.co/learn/llm-course/en/chapter2/1)
- [Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/en/chapter6/1)

## 0. Опрацюйте демо-ноутбук лекції 4

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (якщо потрібно дописати якийсь код, то в блоці "Мої відповіді наведіть відповідний код")

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?
2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))
2. Що таке `"<unk>", "<eos>"` в 2.1.?
3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).
4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?
5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?
6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.
7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?
8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.
10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?
11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.
12. За що саме відповідає параметр `top_p`?
13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

### 0.2. Мої відповіді:

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?

**Змінилося:** BPE-алгоритм шукав би найчастіші пари символів/байтів по всьому суцільному тексту, оскільки вірші часто закінчуються однаковими розділовими знаками або словами, а наступний починається з великої літери, токенізатор міг би злити ці межі в єдині токени.


**Зламалося:** Токенізатор втратив би розуміння того, де закінчується один твір і починається інший. Це призвело б до роздування словника "сміттєвими" токенами, які зустрічаються лише на стиках текстів. Крім того, під час тренування ми б втратили можливість природно вставляти токен <eos> (кінець послідовності) між віршами, і модель сприймала б весь корпус як одне гігантське речення.

2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))

**1 - Нормалізація (Normalization):** Приведення тексту до єдиного вигляду (наприклад, видалення зайвих пробілів, приведення до нижнього регістру, нормалізація Unicode). У демо це частково робить сам BPE, працюючи з байтами.

**2 - Попередня токенізація (Pre-tokenization):** Розбиття тексту на більш дрібні шматки (слова, підслова або окремі байти) перед основним алгоритмом. У демо використовується ByteLevel, який розбиває текст на байти.

**3 - Модель токенізації (Model):** Основний алгоритм, який перетворює ці шматки на токени (у демо - BPE, який ітеративно зливає найчастіші пари байтів).

**4 - Пост-обробка (Post-processing):** Додавання спеціальних токенів (наприклад, `<eos>`, `<unk>`, `<s>`) або форматування для конкретної моделі (наприклад, додавання пробілів).

3. Що таке `"<unk>", "<eos>"` в 2.1.?

`<unk>` (unknown) - це спеціальний токен, який використовується для позначення символів або слів, яких немає у словнику токенізатора. Модель не знає, що це за слово, але розуміє, що там щось було.

`<eos>` (end of sequence) - це спеціальний токен, який позначає кінець тексту або окремого твору. Він допомагає моделі зрозуміти, де закінчується логічна послідовність, і вчитися генерувати завершені тексти.

4. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).

**До зміни:**

Розмір словника (BPE): 16000

Токенів у корпусі (обраний токенізатор): 18421968

Токенів поза словником (`<unk>`): 0.0%

Форма закодованого корпусу: torch.Size([18421968])

**Після зміни:**

Розмір словника (BPE): 6000

Токенів у корпусі (обраний токенізатор): 21041508

Токенів поза словником (`<unk>`): 0.0%

Форма закодованого корпусу: torch.Size([21041508])

**Пояснення:**

Менший словник (6000 замість 16000) означає, що токенізатору доведеться розбивати слова на більше дрібних частин (підслів або навіть окремих байтів). Через це середня кількість токенів на один вірш зросте.

Частка `<unk>` не збільшиться, тому що використовується ByteLevel BPE. Він розбиває текст на байти, а базові байти завжди є у словнику. Тому навіть якщо цілого слова немає, воно збереться з байтів.

Оскільки послідовність токенів стане довшою, при тому ж `CONTEXT_LEN = 64` модель буде "бачити" менше слів (менший контекст), тобто кількість кроків для розуміння тексту зменшиться.

5. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?

Токен `<eos>` використовується як роздільник між окремими віршами. Якщо його прибрати, при конкатенації (склеюванні) текстів останній токен одного вірша зіткнеться з першим токеном наступного. Модель під час тренування бачитиме це як один суцільний текст і навчиться генерувати продовження, які не мають логічного завершення (наприклад, речення одного вірша плавно перетікатиме в речення зовсім іншого). `<eos>` вчить модель зупинятися.

6. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?

**Формат:** Це одномірні тензори PyTorch (тип torch.int64), які є послідовністю цілих чисел.

**Що репрезентують:** Кожне число - це ID токена у словнику BPE. Вони разом утворюють закодований текст усього корпусу поезії, розділений на навчальну (90%) та валідаційну (10%) вибірки.

7. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.

In [1]:
assert xb.shape == (BATCH_SIZE, CONTEXT_LEN), "Неправильна форма xb"
assert yb.shape == (BATCH_SIZE, CONTEXT_LEN), "Неправильна форма yb"
#Перевірка, що y це x, зсунутий на 1 токен вперед
assert torch.equal(xb[:, 1:], yb[:, :-1]), "yb не є зсунутим xb"

NameError: name 'xb' is not defined

8. У 3.3 `causal_mask` - нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?

**Чому нижньотрикутна:** Ми дивимося на поточний токен і всі попередні. Нижньотрикутна матриця дозволяє множити поточний запит (query) на ключі (keys) лише тих токенів, які знаходяться на позиції <= поточної. Верхня трикутна частина заповнюється -inf, тому ймовірність "підглянути" в майбутнє стає нульовою.

**Якби маски не було:** Модель під час тренування бачила б майбутні токени (target). Вона б просто навчилася копіювати наступний токен з входу (завдяки механізму self-attention), і loss впав би до 0. Але під час генерації тексту майбутніх токенів ще не існує, тому модель би видавала повну маячню (адже вона звикла підглядати).

9. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.

**Чекпоінти збігаються:** Якщо модель тренується монотонно, і `val_loss` постійно знижується до самого кінця навчання.

**Чекпоінти НЕ збігаються:** Якщо модель почала перенавчатися (overfitting). Тоді `train_loss` продовжує падати, а `val_loss` починає зростати. Найкращий чекпоінт буде десь посередині.

**У нашому випадку:** Крива `val_loss` плавно знижується до самого кінця (iter 7999), тому найкращий чекпоінт збігається з останнім. Ознак перенавчання не видно (що логічно для такої маленької моделі та великого датасету).

10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?

**В результаті:** Крива стане гладкішою, оскільки ми отримуємо більше точок для побудови графіка. Однак це значно сповільнить тренування.

**Ціна:** Кожен виклик `estimate_loss()` проганяє `EVAL_ITERS` (50) батчів. Якщо робити це кожні 50 ітерацій замість 250, ми витратимо додатково (8000 / 50) * 50 = 8000 форвардних проходів на валідацію (в ідеалі без градієнтів). Це помітно збільшить час навчання (у демо з початковими даними тренування зайняло 6.4 хв, а з таким `EVAL_INTERVAL` воно зайняло 9.9 хв).

**Чи є сенс:** Ні, робити його ще меншим (наприклад, 10) недоцільно. Крива і так буде достатньо гладкою при 50, а витрати часу стануть невиправданими.

11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.

**Математично:** Логіти (сирі оцінки моделі) діляться на temperature (T). probs = softmax(logits / T).

**При T < 1 (наприклад, 0.2):** Різниця між логітами штучно збільшується. Після softmax найбільш ймовірний токен отримує ймовірність, близьку до 1, а решта - близьку до 0. Модель стає дуже впевненою і передбачуваною (майже завжди вибирає найімовірніший токен).

**При T > 1 (наприклад, 1.4):** Різниця між логітами згладжується. Розподіл ймовірностей стає більш плоским (рівномірним). Модель починає частіше вибирати токени, які мали низьку ймовірність, що робить текст креативнішим, але при дуже високих T - майже випадковим.

12. За що саме відповідає параметр `top_p`?

`top_p` відповідає за обмеження вибору токенів за сукупною ймовірністю. Модель сортує токени за спаданням ймовірності і залишає лише ті, сумарна ймовірність яких не перевищує p (наприклад, 0.9). Решта токенів відсікається. Це запобігає вибору зовсім нелогічних токенів з "хвоста" розподілу, навіть якщо температура висока.

13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

**Приклад з `enable_thinking=False`:**

--- temperature=0.8, top_p=0.6 ---

Якось уночі студент Житомирської політехніки прокинувся від дивного шуму. Він підвівся з ліжка, зігнувся, підійшов до вікна і зустрів на підлозі чорну, заспокійливо мовчаливу тварю, яку він не мог визначити.

**Результати з `enable_thinking=True`:**

--- temperature=0.8, top_p=0.6 ---

`<think>`

Okay, the user wants me to continue the sentence: "Якось уночі студент Житомирської політехніки прокинувся від дивного шуму. Він підвівся з ліжка..." So, first, I need to make sure I understand the original sentence correctly. It's a Ukrainian sentence, so I should maintain the language. The main points are: a student from the Kharkiv Polytechnic University (wait, no, it's Житомирської, which is Khmelnytskyi, not Kharkiv) wakes up from a strange noise, gets up from bed. The user wants the continuation.

I need to think about the possible direction of the story. The student is in a dorm, maybe in a university setting. The strange noise could be something unusual, like a mysterious sound, maybe a ghost story, or a supernatural element. Alternatively, it could be a normal but eerie sound, like a creaking floorboard, or something more sinister.

Since it's a continuation, the next part should logically follow. The student gets up, maybe checks the source of the noise. The sentence starts with "Він підвівся з ліжка..." so the next part should describe his actions. Maybe he looks around, hears more sounds, or sees something. The tone should be

**Загально по змінах:**

Тепер у виводі ми бачимо роздуми моделі про те, як продовжити речення. Це може покращити логіку відповіді, але збільшить час генерації та споживання токенів.

---
## 0. Налаштування середовища

### 0.0. Імпортуємо бібліотеки

In [3]:
import sys
import os
import math
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

### 0.1. Перевіряємо чи ми точно в Colab

In [4]:
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

assert IN_COLAB, f"Ця лаба має виконуватися в Google Colab (https://colab.research.google.com/)"

Running in Colab: True


### 0.2. Використовуємо графічний прискорювач, якщо доступний.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Використовуємо:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Використовуємо: cuda
GPU: Tesla T4


### 0.3. Монтуємо Google Drive

In [6]:
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/lab3"
MODELS_DIR = os.path.join(BASE_DIR, "models")
RUNS_DIR = os.path.join(BASE_DIR, "runs")

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

print("Артефакти зберігаються в:", BASE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Артефакти зберігаються в: /content/drive/MyDrive/lab3


---
## 1. Свій датасет

Знайдіть **свій** українськомовний текстовий корпус, достатній за обсягом для mini-GPT — наприклад, на [Hugging Face Datasets](https://huggingface.co/datasets?language=language:uk) або на [Kaggle](https://www.kaggle.com/datasets?search=ukrainian). Жанр і епоха — на ваш вибір (новини, художня література, вікіпедія, форуми, судові рішення тощо), головне — одна мова (українська) і достатньо тексту, щоб токенізатор і модель побачили бодай прості закономірності.

⚠️ Якщо датасет великий (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`), інакше отримаєте помилку нестачі пам'яті (out of memory, OOM).

⚠️ Якщо датасет доступний лише через `streaming=True` (наприклад, через розмір) — `load_dataset(..., streaming=True).take(N)` повертає `IterableDataset`, у якого **нема** `.to_pandas()` і `len()`. Спершу матеріалізуйте вибірку в список, а вже потім — у `DataFrame`:
```python
rows = list(streamed_ds)
poems_df = pd.DataFrame(rows)
```

### 1.1. Завантажуємо датасет з Hugging Face Hub

In [7]:
from datasets import load_dataset
import pandas as pd

poems_dataset = load_dataset("shamotskyi/ua_cbt_stories", split="train")
print("Усього оповідань у датасеті:", len(poems_dataset))
print("Колонки:", poems_dataset.column_names)

README.md:   0%|          | 0.00/4.30k [00:00<?, ?B/s]

dataset.csv:   0%|          | 0.00/1.95M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/133 [00:00<?, ? examples/s]

Усього оповідань у датасеті: 133
Колонки: ['story_id', 'story_template', '1_llm_generated', '2_llm_expanded', '3_llm_corrected', '4_human_corrected', 'usable', 'whynot', 'annotator_comments', 'annotator', 'notable', 'UNHAPPY_ENDING', 'N_MAIN_CHARACTERS', 'N_MINOR_CHARACTERS', 'model_generation', 'model_correction']


### 1.2. Збираємо суцільний корпус

In [8]:
poems_df = poems_dataset.to_pandas()
corpus_text = "\n\n".join(poems_df["4_human_corrected"].dropna().astype(str).tolist())

print("Розмір корпусу:", len(corpus_text), "символів")
print("Перші 500 символів корпусу:\n", corpus_text[:500])

Розмір корпусу: 293226 символів
Перші 500 символів корпусу:
 Одного разу, на березі величезного озера, де сонце ласкаво цілувало гладку поверхню, жила Черепаха, славна своєю лінню та невмілістю. Її життя текло повільно, як і кожен її крок, але в глибині душі Черепаха мала таємницю: вона була вправною кравчинею, хоча ніхто про це не знав.

Біля озера також мешкали Лисиця, завжди в пошуках найкращого одягу для своїх хитрих планів, і Заєць, який постійно рвав свій одяг, стрибаючи крізь кущі й камені. Черепаха спостерігала за ними з-під свого твердого панцирн


### 1.3. Базова статистика корпусу

In [9]:
words = corpus_text.split()
unique_words = set(w.lower() for w in words)

print("Слів (з повторами):   ", len(words))
print("Унікальних слів:      ", len(unique_words))
print("Приклад уривку:\n")
print(corpus_text[:500])

Слів (з повторами):    45603
Унікальних слів:       12097
Приклад уривку:

Одного разу, на березі величезного озера, де сонце ласкаво цілувало гладку поверхню, жила Черепаха, славна своєю лінню та невмілістю. Її життя текло повільно, як і кожен її крок, але в глибині душі Черепаха мала таємницю: вона була вправною кравчинею, хоча ніхто про це не знав.

Біля озера також мешкали Лисиця, завжди в пошуках найкращого одягу для своїх хитрих планів, і Заєць, який постійно рвав свій одяг, стрибаючи крізь кущі й камені. Черепаха спостерігала за ними з-під свого твердого панцирн


---
## 2. Токенізатори: BPE, WordPiece, Unigram

Демо тренує лише BPE. У цій секції ви натренуєте й порівняєте всі три основні алгоритми токенізації на **своєму** тексті.

Матеріал — офіційний [Hugging Face LLM Course, розділ 6](https://huggingface.co/learn/llm-course/en/chapter6/1). Пройдіть і повторіть на своїх даних:

1. **[6.2. Тренування нового токенізатора на основі старого](https://huggingface.co/learn/llm-course/en/chapter6/2)** — найпростіший шлях: `tokenizer.train_new_from_iterator(...)` на основі архітектури готового токенізатора (наприклад, `Qwen/Qwen2.5-0.5B-Instruct`). Зробіть це як розминку перед ручними варіантами нижче.
2. **[6.5. Алгоритм BPE](https://huggingface.co/learn/llm-course/en/chapter6/5)** — повторіть покрокову реалізацію з туторіалу на своєму тексті, повністю вручну або за допомогою бібліотеки `tokenizers`.
3. **[6.6. Алгоритм WordPiece](https://huggingface.co/learn/llm-course/en/chapter6/6)** — так само.
4. **[6.7. Алгоритм Unigram](https://huggingface.co/learn/llm-course/en/chapter6/7)** — так само.

### 1.1. Порівняння

Для однакового `vocab_size` (візьміть, наприклад, 500 — на невеликому зразку тексту більший словник не встигне показати різницю) порівняйте три токенізатори в невеликій таблиці: розмір словника, кількість токенів на вашому зразку, скільки токенів виходить на кількох обраних словах (частотне й рідкісне — як у лекції). Коротко письмово: який алгоритм дав найкоротші послідовності на вашому тексті і чи це узгоджується з тим, що каже туторіал про природу кожного алгоритму?

### 2.0. Підготовлюємо невеликий фрагмент нашого корпусу (перші 1000 оповідань)

In [10]:
import gc
import numpy as np
from collections import defaultdict
from math import log
import copy
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import AutoTokenizer

#Беремо підвибірку для швидкості ручних алгоритмів
sample_texts = poems_df["4_human_corrected"].dropna().astype(str).tolist()[:1000]
sample_corpus = "\n\n".join(sample_texts)

#Розбиваємо на окремі слова для ручних алгоритмів
word_freqs = defaultdict(int)
for text in sample_texts:
    for word in text.split():
        if word:  #ігноруємо пусті
            word_freqs[word] += 1

print(f"Кількість унікальних слів у вибірці: {len(word_freqs)}")

Кількість унікальних слів у вибірці: 12721


### 2.1. Тренування нового токенізатора на основі старого

In [11]:
old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

def get_training_corpus():
    for i in range(0, len(sample_texts), 100):
        yield sample_texts[i:i+100]

tokenizer_hf = old_tokenizer.train_new_from_iterator(get_training_corpus(), 500)

example_text = sample_texts[0][:200]
tokens_hf = tokenizer_hf.tokenize(example_text)
print(f"Приклад токенізації (train_new_from_iterator):\n{tokens_hf[:20]}...")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Приклад токенізації (train_new_from_iterator):
['Ð', 'ŀ', 'Ð´Ð½Ð¾Ð³Ð¾', 'ĠÑĢÐ°Ð·', 'Ñĥ', ',', 'ĠÐ½Ð°', 'ĠÐ±', 'ÐµÑĢÐµ', 'Ð·', 'Ñĸ', 'ĠÐ²', 'ÐµÐ»Ð¸', 'ÑĩÐµ', 'Ð·', 'Ð½Ð¾Ð³Ð¾', 'ĠÐ¾', 'Ð·', 'ÐµÑĢ', 'Ð°']...


### 2.2. Алгоритм BPE

In [12]:
#Початковий алфавіт
alphabet = set()
for word in word_freqs.keys():
    for letter in word:
        alphabet.add(letter)
alphabet = sorted(list(alphabet))

vocab_bpe = ["<unk>", "<eos>"] + alphabet.copy()

#Початкова розбивка слів на символи
splits_bpe = {word: [c for c in word] for word in word_freqs.keys()}

#3. Функція підрахунку частот пар
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

#Функція злиття пари
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue
        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2:]
            else:
                i += 1
        splits[word] = split
    return splits

#Тренування BPE до розміру словника 500
VOCAB_SIZE = 500
merges_bpe = {}

while len(vocab_bpe) < VOCAB_SIZE:
    pair_freqs = compute_pair_freqs(splits_bpe)
    if not pair_freqs:
        break
    best_pair = max(pair_freqs, key=pair_freqs.get)
    splits_bpe = merge_pair(*best_pair, splits_bpe)
    merges_bpe[best_pair] = best_pair[0] + best_pair[1]
    vocab_bpe.append(best_pair[0] + best_pair[1])

#Функція токенізації для BPE
def tokenize_bpe(text):
    words = text.split()
    encoded_words = []
    for word in words:
        splits = [[l for l in word]]
        for pair, merge in merges_bpe.items():
            for idx, split in enumerate(splits):
                i = 0
                while i < len(split) - 1:
                    if split[i] == pair[0] and split[i + 1] == pair[1]:
                        split = split[:i] + [merge] + split[i + 2:]
                    else:
                        i += 1
                splits[idx] = split
        encoded_words.extend(sum(splits, []))
    return encoded_words

tokens_bpe = tokenize_bpe(example_text)

print(f"Приклад токенізації (BPE):\n{tokens_bpe[:20]}...")

Приклад токенізації (BPE):
['Одного', 'разу', ',', 'на', 'б', 'ере', 'зі', 'вели', 'че', 'з', 'ного', 'о', 'з', 'ер', 'а', ',', 'де', 'со', 'н', 'це']...


### 2.3. Алгоритм WordPiece

In [13]:
#Початковий алфавіт з префіксами
alphabet_wp = set()
for word in word_freqs.keys():
    if len(word) > 0:
        alphabet_wp.add(word[0])
        for letter in word[1:]:
            alphabet_wp.add(f"##{letter}")
alphabet_wp = sorted(list(alphabet_wp))

vocab_wp = ["<unk>", "<eos>"] + alphabet_wp.copy()

#Розбивка слів з префіксами
splits_wp = {
    word: [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
    for word in word_freqs.keys()
}

#Підрахунок скорів
def compute_pair_scores(splits):
    letter_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            letter_freqs[split[0]] += freq
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            letter_freqs[split[i]] += freq
            pair_freqs[pair] += freq
        letter_freqs[split[-1]] += freq

    scores = {
        pair: freq / (letter_freqs[pair[0]] * letter_freqs[pair[1]])
        for pair, freq in pair_freqs.items()
    }
    return scores

#Злиття з урахуванням префікса
def merge_pair_wp(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue
        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                merge = a + b[2:] if b.startswith("##") else a + b
                split = split[:i] + [merge] + split[i + 2:]
            else:
                i += 1
        splits[word] = split
    return splits

#Тренування WordPiece
while len(vocab_wp) < VOCAB_SIZE:
    scores = compute_pair_scores(splits_wp)
    if not scores:
        break
    best_pair = max(scores, key=scores.get)
    splits_wp = merge_pair_wp(*best_pair, splits_wp)
    new_token = (
        best_pair[0] + best_pair[1][2:]
        if best_pair[1].startswith("##")
        else best_pair[0] + best_pair[1]
    )
    vocab_wp.append(new_token)

#Токенізація для WordPiece
def encode_word_wp(word, vocab):
    tokens = []
    while len(word) > 0:
        i = len(word)
        while i > 0 and word[:i] not in vocab:
            i -= 1
        if i == 0:
            return ["<unk>"]
        tokens.append(word[:i])
        word = word[i:]
        if len(word) > 0:
            word = f"##{word}"
    return tokens

def tokenize_wp(text):
    words = text.split()
    encoded_words = [encode_word_wp(word, vocab_wp) for word in words]
    return sum(encoded_words, [])

tokens_wp = tokenize_wp(example_text)
print(f"Приклад токенізації (WordPiece):\n{tokens_wp[:20]}...")

Приклад токенізації (WordPiece):
['Одн', '##о', '##г', '##о', 'р', '##а', '##з', '##у', '##,', 'н', '##а', 'б', '##е', '##р', '##е', '##з', '##і', 'в', '##е', '##л']...


### 2.4. Алгоритм Unigram

In [14]:
#Початковий словник (символи + найчастіші підрядки)
char_freqs = defaultdict(int)
subwords_freqs = defaultdict(int)

for word, freq in word_freqs.items():
    for i in range(len(word)):
        char_freqs[word[i]] += freq
        for j in range(i + 2, len(word) + 1):
            subwords_freqs[word[i:j]] += freq

sorted_subwords = sorted(subwords_freqs.items(), key=lambda x: x[1], reverse=True)

token_freqs = list(char_freqs.items()) + sorted_subwords[:VOCAB_SIZE - len(char_freqs)]
token_freqs = {token: freq for token, freq in token_freqs}

#Підрахунок log-ймовірностей
total_sum = sum(token_freqs.values())
model_uni = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}

#Функція Вітербі для сегментації
def encode_word_uni(word, model):
    best_segmentations = [{"start": 0, "score": 1}] + [
        {"start": None, "score": None} for _ in range(len(word))
    ]
    for start_idx in range(len(word)):
        best_score_at_start = best_segmentations[start_idx]["score"]
        for end_idx in range(start_idx + 1, len(word) + 1):
            token = word[start_idx:end_idx]
            if token in model and best_score_at_start is not None:
                score = model[token] + best_score_at_start
                if (
                    best_segmentations[end_idx]["score"] is None
                    or best_segmentations[end_idx]["score"] > score
                ):
                    best_segmentations[end_idx] = {"start": start_idx, "score": score}

    segmentation = best_segmentations[-1]
    if segmentation["score"] is None:
        return ["<unk>"], None

    score = segmentation["score"]
    start = segmentation["start"]
    end = len(word)
    tokens = []
    while start != 0:
        tokens.insert(0, word[start:end])
        next_start = best_segmentations[start]["start"]
        end = start
        start = next_start
    tokens.insert(0, word[start:end])
    return tokens, score

#Функція підрахунку loss
def compute_loss_uni(model):
    loss = 0
    for word, freq in word_freqs.items():
        _, word_loss = encode_word_uni(word, model)
        loss += freq * word_loss
    return loss

#Функція підрахунку скорів для видалення
def compute_scores_uni(model):
    scores = {}
    model_loss = compute_loss_uni(model)
    for token, score in model.items():
        if len(token) == 1:
            continue
        model_without_token = copy.deepcopy(model)
        _ = model_without_token.pop(token)
        scores[token] = compute_loss_uni(model_without_token) - model_loss
    return scores

#Тренування (видалення токенів)
percent_to_remove = 0.1
while len(model_uni) > VOCAB_SIZE:
    scores = compute_scores_uni(model_uni)
    sorted_scores = sorted(scores.items(), key=lambda x: x[1])
    for i in range(int(len(model_uni) * percent_to_remove)):
        if i < len(sorted_scores):
            token_freqs.pop(sorted_scores[i][0], None)

    total_sum = sum(token_freqs.values())
    model_uni = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}

#Токенізація
def tokenize_uni(text, model):
    words = text.split()
    encoded_words = [encode_word_uni(word, model)[0] for word in words]
    return sum(encoded_words, [])

tokens_uni = tokenize_uni(example_text, model_uni)
print(f"Приклад токенізації (Unigram):\n{tokens_uni[:20]}...")

Приклад токенізації (Unigram):
['О', 'д', 'ного', 'раз', 'у,', 'на', 'б', 'ере', 'зі', 'в', 'ели', 'че', 'з', 'ного', 'оз', 'ер', 'а,', 'де', 'с', 'он']...


### 1.1. Порівняння

In [15]:
test_text = sample_corpus[:5000]
words_test = test_text.split()

tokens_bpe_test = tokenize_bpe(test_text)
tokens_wp_test = tokenize_wp(test_text)
tokens_uni_test = tokenize_uni(test_text, model_uni)

avg_bpe = len(tokens_bpe_test) / len(words_test)
avg_wp = len(tokens_wp_test) / len(words_test)
avg_uni = len(tokens_uni_test) / len(words_test)

unk_bpe = tokens_bpe_test.count("<unk>") / len(tokens_bpe_test)
unk_wp = tokens_wp_test.count("<unk>") / len(tokens_wp_test)
unk_uni = tokens_uni_test.count("<unk>") / len(tokens_uni_test)

print(f"{'Алгоритм':<15} | {'Розмір словника':<15} | {'Токенів/слово':<15} | {'Частка <unk>':<15}")
print("-" * 65)
print(f"{'BPE':<15} | {len(vocab_bpe):<15} | {avg_bpe:<15.2f} | {unk_bpe:<15.1%}")
print(f"{'WordPiece':<15} | {len(vocab_wp):<15} | {avg_wp:<15.2f} | {unk_wp:<15.1%}")
print(f"{'Unigram':<15} | {len(model_uni):<15} | {avg_uni:<15.2f} | {unk_uni:<15.1%}")

Алгоритм        | Розмір словника | Токенів/слово   | Частка <unk>   
-----------------------------------------------------------------
BPE             | 500             | 2.44            | 0.0%           
WordPiece       | 500             | 5.36            | 0.0%           
Unigram         | 500             | 2.73            | 0.0%           


**Висновки:**

Найкоротші послідовності дав алгоритм BPE так як він зливає найчастіші пари байтів/символів, що призводить до більш "компактного" представлення частих слів.

І так, це узгоджується це з теорією. У туторіалі зазначено, що BPE створює токени, орієнтуючись на частоту пар, тому він добре стискає текст. WordPiece використовує скоринг, який може бути менш агресивним. Unigram починає з великого словника і видаляє рідкісні токени, тому він може залишати більше дрібних частин для рідкісних слів.

---
## 3. mini-GPT на своїх даних

Повторіть підхід із розділів 2-3 демо-ноутбука на своєму датасеті — не копіюйте код один в один: у вашого датасету інша колонка з текстом, інший обсяг, інший жанр/регістр мови — токенізатор і модель побачать зовсім інші закономірності, ніж на поезії з демо.

### 3.1. Корпус і токенізація

1. Завантажте свій датасет і зберіть `corpus_text` з потрібної колонки (демо 1.2).
2. Натренуйте токенізатор на своєму корпусі (демо 2.1). **Якщо ваш датасет великий** (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`). Не забудьте `<eos>` між документами (демо 2.2).
3. Спліт train/val (демо 2.3).

### 3.2. Модель: щонайменше 2 варіанти

За зразком розділу 3 демо (`AttentionHead`, `MultiHeadAttention`, `TransformerBlock`, `MiniGPT`) — тренуйте щонайменше 2 варіанти, які відрізняються параметрами: інший `VOCAB_SIZE`, інша `CONTEXT_LEN`, інша кількість шарів/голів (`N_LAYERS`/`N_HEADS`), або інший `MAX_ITERS`. Кожен варіант логуйте в TensorBoard в окрему підпапку (як у демо 3.8), з осмисленою назвою.

### 3.3. Генерація і висновки

Для кожного варіанта — генерація на найкращому за `val_loss` чекпоінті (демо 3.10), той самий промпт для всіх варіантів, щоб порівняння було чесним. Коротко письмово: який варіант дав кращий `val_loss` і чи помітно це за якістю згенерованого тексту? Що саме (зміна словника, глибини мережі, довжини контексту) дало найбільший ефект?

### 3.1. Корпус і токенізація

In [16]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import numpy as np
import gc
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from torch.utils.tensorboard import SummaryWriter

MODELS_DIR = "/content/drive/MyDrive/lab3/models"
RUNS_DIR = "/content/drive/MyDrive/lab3/runs"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

#--- Токенізація ---
VOCAB_SIZE = 8000
SPECIAL_TOKENS = ["<unk>", "<eos>"]

bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()

bpe_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)

trainer_iterator = poems_df["4_human_corrected"].dropna().astype(str).tolist()
bpe_tokenizer.train_from_iterator(trainer_iterator, trainer=bpe_trainer)
del trainer_iterator
gc.collect()

bpe_vocab = bpe_tokenizer.get_vocab()
itos = [None] * len(bpe_vocab)
for tok, idx in bpe_vocab.items():
    itos[idx] = tok
stoi = bpe_vocab
UNK_ID = stoi["<unk>"]
EOS_ID = stoi["<eos>"]

def tokenize(text: str) -> list[str]:
    return bpe_tokenizer.encode(text).tokens

def encode(tokens: list[str]) -> list[int]:
    return [stoi.get(t, UNK_ID) for t in tokens]

def decode(ids: list[int]) -> str:
    return bpe_tokenizer.decode(ids)

print("Розмір словника (BPE):", len(itos))

#--- Кодуємо корпус ---
print("Кодуємо корпус...")
ENCODE_BATCH_SIZE = 1000
poems_list = poems_df["4_human_corrected"].dropna().astype(str).tolist()

token_id_chunks = []
for start in range(0, len(poems_list), ENCODE_BATCH_SIZE):
    batch = poems_list[start:start + ENCODE_BATCH_SIZE]
    encodings = bpe_tokenizer.encode_batch(batch)
    for enc in encodings:
        token_id_chunks.append(np.array(enc.ids + [EOS_ID], dtype=np.int32))
    del encodings
del poems_list
gc.collect()

token_ids = np.concatenate(token_id_chunks)
del token_id_chunks
gc.collect()

data = torch.from_numpy(token_ids.astype(np.int64))
print("Форма закодованого корпусу:", data.shape)

#--- Спліт train/val ---
split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
val_data = data[split_idx:]
print("train:", train_data.shape, " val:", val_data.shape)

Розмір словника (BPE): 8000
Кодуємо корпус...
Форма закодованого корпусу: torch.Size([67887])
train: torch.Size([61098])  val: torch.Size([6789])


### 3.2. Модель: щонайменше 2 варіанти

In [17]:
class AttentionHead(nn.Module):
    def __init__(self, embed_dim: int, head_dim: int, context_len: int, dropout: float):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        causal_mask = torch.tril(torch.ones(context_len, context_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        return weights @ v

#Multi-head attention
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_len: int, dropout: float):
        super().__init__()
        head_dim = embed_dim // n_heads
        self.heads = nn.ModuleList([AttentionHead(embed_dim, head_dim, context_len, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))

#Блок трансформера
class FeedForward(nn.Module):
    def __init__(self, embed_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_len: int, dropout: float):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, n_heads, context_len, dropout)
        self.feed_forward = FeedForward(embed_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x

#Повна модель
class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, n_heads: int, n_layers: int, context_len: int, dropout: float):
        super().__init__()
        self.context_len = context_len
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(context_len, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, n_heads, context_len, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0, top_k: int | None = None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

Функція для тренування (щоб не дублювати код)

In [18]:
def train_model(config: dict):
    print(f"\n--- Тренування {config['name']} ---")

    def get_batch(split: str):
        source = train_data if split == "train" else val_data
        ix = torch.randint(len(source) - config["context_len"] - 1, (config["batch_size"],))
        x = torch.stack([source[i:i + config["context_len"]] for i in ix])
        y = torch.stack([source[i + 1:i + config["context_len"] + 1] for i in ix])
        return x.to(device), y.to(device)

    @torch.no_grad()
    def estimate_loss(model):
        model.eval()
        out = {}
        for split in ("train", "val"):
            losses = torch.zeros(config["eval_iters"])
            for i in range(config["eval_iters"]):
                xb, yb = get_batch(split)
                _, loss = model(xb, yb)
                losses[i] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

    model = MiniGPT(
        vocab_size=len(itos),
        embed_dim=config["embed_dim"],
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        context_len=config["context_len"],
        dropout=config["dropout"],
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Кількість параметрів: {n_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
    writer = SummaryWriter(log_dir=os.path.join(RUNS_DIR, config["name"]))

    best_ckpt_path = os.path.join(MODELS_DIR, f"{config['name']}.pt")

    best_val_loss = float("inf")
    start_time = time.time()

    for it in range(config["max_iters"]):
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if it % config["eval_interval"] == 0 or it == config["max_iters"] - 1:
            losses = estimate_loss(model)
            writer.add_scalar("Loss/train", losses["train"], it)
            writer.add_scalar("Loss/val", losses["val"], it)

            elapsed_min = (time.time() - start_time) / 60
            print(f"iter {it:5d}  train_loss={losses['train']:.3f}  val_loss={losses['val']:.3f}  ({elapsed_min:.1f} хв)")

            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                torch.save(model.state_dict(), best_ckpt_path)

    writer.close()
    print(f"Тренування {config['name']} завершено за {(time.time() - start_time) / 60:.1f} хв")
    print(f"Найкращий val_loss: {best_val_loss:.3f}")

    return best_ckpt_path

Запуск тренування

In [19]:
config_small = {
    "name": "minigpt_v1_small",
    "context_len": 64,
    "embed_dim": 128,
    "n_heads": 4,
    "n_layers": 4,
    "dropout": 0.1,
    "batch_size": 64,
    "learning_rate": 3e-4,
    "max_iters": 3000,
    "eval_interval": 250,
    "eval_iters": 50,
}

path_v1 = train_model(config_small)


--- Тренування minigpt_v1_small ---
Кількість параметрів: 2,856,000
iter     0  train_loss=9.051  val_loss=9.056  (0.0 хв)
iter   250  train_loss=6.328  val_loss=6.611  (0.2 хв)
iter   500  train_loss=5.458  val_loss=6.246  (0.4 хв)
iter   750  train_loss=4.729  val_loss=6.149  (0.5 хв)
iter  1000  train_loss=4.109  val_loss=6.184  (0.7 хв)
iter  1250  train_loss=3.573  val_loss=6.301  (0.8 хв)
iter  1500  train_loss=3.128  val_loss=6.459  (0.9 хв)
iter  1750  train_loss=2.758  val_loss=6.633  (1.1 хв)
iter  2000  train_loss=2.411  val_loss=6.816  (1.2 хв)
iter  2250  train_loss=2.088  val_loss=7.058  (1.3 хв)
iter  2500  train_loss=1.798  val_loss=7.276  (1.5 хв)
iter  2750  train_loss=1.526  val_loss=7.480  (1.6 хв)
iter  2999  train_loss=1.280  val_loss=7.737  (1.8 хв)
Тренування minigpt_v1_small завершено за 1.8 хв
Найкращий val_loss: 6.149


In [20]:
config_medium = {
    "name": "minigpt_v2_medium",
    "context_len": 128,
    "embed_dim": 256,
    "n_heads": 8,
    "n_layers": 6,
    "dropout": 0.1,
    "batch_size": 64,
    "learning_rate": 5e-4,
    "max_iters": 10000,
    "eval_interval": 500,
    "eval_iters": 50,
}

path_v2 = train_model(config_medium)


--- Тренування minigpt_v2_medium ---
Кількість параметрів: 8,871,232
iter     0  train_loss=8.671  val_loss=8.652  (0.1 хв)
iter   500  train_loss=2.368  val_loss=6.590  (1.7 хв)
iter  1000  train_loss=0.267  val_loss=8.233  (3.2 хв)
iter  1500  train_loss=0.116  val_loss=9.094  (4.8 хв)
iter  2000  train_loss=0.087  val_loss=9.518  (6.3 хв)
iter  2500  train_loss=0.074  val_loss=9.933  (7.9 хв)
iter  3000  train_loss=0.067  val_loss=10.175  (9.4 хв)
iter  3500  train_loss=0.061  val_loss=10.485  (11.0 хв)
iter  4000  train_loss=0.059  val_loss=10.716  (12.5 хв)
iter  4500  train_loss=0.057  val_loss=10.943  (14.1 хв)
iter  5000  train_loss=0.054  val_loss=11.222  (15.6 хв)
iter  5500  train_loss=0.052  val_loss=11.378  (17.2 хв)
iter  6000  train_loss=0.051  val_loss=11.634  (18.7 хв)
iter  6500  train_loss=0.050  val_loss=11.866  (20.3 хв)
iter  7000  train_loss=0.048  val_loss=12.068  (21.8 хв)
iter  7500  train_loss=0.047  val_loss=12.216  (23.4 хв)
iter  8000  train_loss=0.047  v

### 3.3. Генерація і висновки

In [33]:
def generate_text(model_path: str, prompt: str, config: dict, max_new_tokens: int = 50) -> str:
    model = MiniGPT(
        vocab_size=len(itos),
        embed_dim=config["embed_dim"],
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        context_len=config["context_len"],
        dropout=config["dropout"],
    ).to(device)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    prompt_ids = torch.tensor([encode(tokenize(prompt))], dtype=torch.long).to(device)
    out_ids = model.generate(prompt_ids, max_new_tokens, temperature=0.8, top_k=40)
    return decode(out_ids[0].tolist())

PROMPT = "Одного разу"

print("=== Варіант 1 (Small) ===")
print(generate_text(path_v1, PROMPT, config_small))
print("\n=== Варіант 2 (Medium) ===")
print(generate_text(path_v2, PROMPT, config_medium))

=== Варіант 1 (Small) ===
Одного разу, коли сонце палило мені за те, як вона була дуже вдячні.

Метелик були вражені навколо на полювання і лисиця: "Але, але нічого не змогла допомогти".

— Ти ж, що заєць, що я не зміг тобі, яка

=== Варіант 2 (Medium) ===
Одного разу в лісі. Лисиця повільно підвівся і їжак ніколи не знали, що робити.

Вовк був дуже вдячний черепасі. Він подякував сові і сказав, що ніколи не знав, що робити.

"Це моя хаткa," - "Я ніколи не брав


**Висновки:**

Кращий `val_loss` дав варіант, оскільки він має більше параметрів, що дозволяє йому вивчати складніші закономірності, також більший `context_len` дає йому більше інформації для передбачення наступного слова.

І так, це помітно за якістю згенерованого тексту (хоч він не дуже зв'язний і в обох випадках). Перший варіант генерує практично беззмістовні фрази, а другий щось схоже на справжні речення.

Найбільший ефект дає збільшення embed_dim (розмірності) та n_layers (глибини мережі).

---
## 4. Playground готових моделей

Тур по задачах через `pipeline()` — на основі [HF LLM Course, Chapter 1, §3](https://huggingface.co/learn/llm-course/en/chapter1/3).

_Зверніть увагу_: в межах попередніх завдань Ви могли завантажити памʼять "під завʼязку", тому для безпроблемного виконання наступних секцій вивільніть памʼять:

1. **Спершу збережіть усе потрібне на диск** — чекпоінти mini-GPT (`models/minigpt_v*.pt`), токенізатори, `results.csv`. Секції 4-6 від змінних секцій 1-3 не залежать, тож після цього їх можна сміливо видаляти.
2. **Видаліть великі обʼєкти** — моделі, оптимізатори, тензори `train_data`/`val_data`, корпус і `DataFrame`-и — і зберіть сміття:
   ```python
   import gc, torch

   del model, optimizer, train_data, val_data, corpus_text  # підставте свої імена змінних
   gc.collect()
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
   ```
   Памʼять звільниться лише тоді, коли на обʼєкт не лишиться **жодного** посилання: зверніть увагу на копії в інших змінних, списках і словниках (наприклад, словник із кількома варіантами моделей).
3. **Перевірте, що памʼять справді звільнилась**: індикатор RAM/GPU вгорі праворуч.
4. **У самій секції 4 не тримайте всі pipeline-и одночасно**: кожен `pipeline(...)` завантажує окрему модель. Перевикористовуйте одну змінну (як `classifier` нижче) або робіть `del` + `gc.collect()` після кожної задачі.
5. **Стежте за диском**: моделі кешуються в `~/.cache/huggingface/hub`, і кілька моделей для summarization/translation легко займуть кілька ГБ. Переглянути й почистити кеш: `hf cache ls` / `hf cache rm <id моделі>`.

In [36]:
import gc
import torch

try:
    del train_data, val_data, corpus_text
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Працюємо на CPU. Пам'ять очищено.")

GPU: зайнято 0.02 GB


### 4.0. Запустіть приклади виконання різних NLP задач

In [1]:
!pip install "transformers[sentencepiece]==5.18.0"

In [2]:
# Sentiment analysis
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I've been waiting for a HuggingFace course my whole life."))
print(classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598046541213989}]
[{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [3]:
# Zero-shot classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.844595193862915, 0.11197695881128311, 0.04342786595225334]}

In [4]:
# Text generation
from transformers import pipeline

generator = pipeline("text-generation")
print(generator("In this course, we will teach you how to"))

generator = pipeline("text-generation", model="distilgpt2")
print(generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
))

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

... (15 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'In this course, we will teach you how to make the necessary changes that are necessary for our customers.\n\n\n\n\nWe will begin with the basics of the data-driven services needed to make our customers more efficient in the business, as well as making them more efficient in the consumer, if not more efficient in the business.\nWe will also introduce a simple, modular approach to the data-driven services we use to communicate with customers.\nWe will also introduce a smart, modular approach to the data-driven services we use to communicate with customers.\nWe will also introduce a smart, modular approach to the data-driven services we use to communicate with customers.\nWe will also introduce a smart, modular approach to the data-driven services we use to communicate with customers.\nIn our course, we will present our approach to data-driven services that make customers more efficient in the business, as well as making them more efficient in the business.\nWe will a

In [5]:
# Fill-mask
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'score': 0.19619743525981903,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.04052695631980896,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [6]:
# Named entity recognition
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [7]:
# Question answering
from transformers import pipeline

question_answerer = pipeline("table-question-answering")
question_answerer(
    query="How many stars does the transformers repository have?",
    table={"Repository":["Transformers","Datasets","Tokenizers"],"Stars":["36542","4512","3934"],"Contributors":["651","77","34"],"Programming language":["Python","Python","Rust, Python and NodeJS"]},
)

[transformers] No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision e3dde19.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/490 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/262k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

{'answer': 'AVERAGE > 36542',
 'coordinates': [(0, 1)],
 'cells': ['36542'],
 'aggregator': 'AVERAGE'}

### 4.1. Свої приклади

Повторіть **усі 6 задач** вище (sentiment-analysis, zero-shot-classification, text-generation, fill-mask, NER, table-question-answering), але замість дефолтних англійських прикладів — придумайте й використайте **свої українськомовні приклади** для кожної задачі (якщо потрібно, підберіть відповідні моделі зі знанням української мови). Для виконання цього завдання постворюйте клітинки нижче.

In [3]:
import gc
import torch

try:
    del train_data, val_data, corpus_text
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Працюємо на CPU. Пам'ять очищено.")

GPU: зайнято 0.00 GB


In [1]:
import gc
import torch

try:
    del train_data, val_data, corpus_text
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Працюємо на CPU. Пам'ять очищено.")

Працюємо на CPU. Пам'ять очищено.


In [2]:
!pip install "huggingface-hub<2.0"

  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 2.4.0
    Uninstalling huggingface_hub-2.4.0:
      Successfully uninstalled huggingface_hub-2.4.0


In [4]:
!pip install transformers[sentencepiece]==4.46.3

In [6]:
!pip install gliclass

In [11]:
#Sentiment Analysis (Аналіз тональності)
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="tabularisai/multilingual-sentiment-analysis")

print(classifier("Я дуже радий, що нарешті закінчив цю лабораторну роботу!"))
print(classifier("Цей день був жахливим, все йшло не так."))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'Positive', 'score': 0.6041914224624634}]
[{'label': 'Negative', 'score': 0.6984800696372986}]


In [8]:
#Zero-shot classification (Класифікація без навчання)
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer
import torch

model_id = "knowledgator/gliclass-multilang-mini"
model = GLiClassModel.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
classifier = ZeroShotClassificationPipeline(
    model, tokenizer, classification_type='multi-label', device=device
)

text = "Ця лабораторна робота присвячена вивченню трансформерів та мовних моделей."
labels = ["освіта", "політика", "бізнес", "технології"]

results = classifier(text, labels, threshold=0.5)[0]

for result in results:
    print(f"{result['label']} => {result['score']:.3f}")

Loading weights:   0%|          | 0/227 [00:00<?, ?it/s]

[transformers] You are using a model of type `GLiClass` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
100%|██████████| 1/1 [00:02<00:00,  2.97s/it]

освіта => 0.963


In [12]:
#Text generation (Генерація тексту)
generator = pipeline("text-generation", model="ai-forever/mGPT")
print(generator("Цього року в університеті ми вивчаємо", max_length=100, num_return_sequences=1, truncation=True))

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.45GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/mGPT
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'Цього року в університеті ми вивчаємо «Музичне мистецтво і культура виховання». Мета і завдання цього курсу – розширити свій кругозір, поповнити його знаннями та досвідом, а також вивчити інструменти та методи музичної освіти'}]


In [47]:
#Fill-mask (Заповнення пропусків)
unmasker = pipeline("fill-mask", model="youscan/ukr-roberta-base")
unmasker("Сьогодні такий <mask> день", top_k=2)

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: youscan/ukr-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.2711363434791565,
  'token': 46349,
  'token_str': ' сонячний',
  'sequence': 'Сьогодні такий  сонячний день'},
 {'score': 0.05944385379552841,
  'token': 12659,
  'token_str': ' особливий',
  'sequence': 'Сьогодні такий  особливий день'}]

In [13]:
#Named Entity Recognition (Розпізнавання іменованих сутностей)
ner = pipeline("ner", aggregation_strategy="simple", model="Davlan/distilbert-base-multilingual-cased-ner-hrl")
ner("Мене звати Олена, і я навчаюся в Житомирі.")

config.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  539MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'entity_group': 'PER',
  'score': np.float32(0.99894786),
  'word': 'Олена',
  'start': 11,
  'end': 16},
 {'entity_group': 'LOC',
  'score': np.float32(0.9959364),
  'word': 'Житомирі',
  'start': 33,
  'end': 41}]

In [16]:
#Question answering (Відповіді на питання)
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name = "robinhad/ukrainian-qa"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

question = "Де живе Сара?"
context = "Мене звати Сара, і я живу в Африці. Африка - це другий за площею і населенням материк у світі."

inputs = tokenizer(question, context, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits) + 1

answer = tokenizer.decode(inputs["input_ids"][0][start_idx:end_idx], skip_special_tokens=True)

print(f"Питання: {question}")
print(f"Відповідь: {answer}")

tokenizer_config.json:   0%|          | 0.00/469 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B /  854kB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.37M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Питання: Де живе Сара?
Відповідь: в Африці


---
## 5. Порівняння моделей

### 5.1. Оберіть задачу й метрику

Візьміть **одну** задачу зі списку нижче:

| Задача | Метрика | Що потрібно підготувати | Метрика в `evaluate` |
|---|---|---|---|
| sentiment-analysis | Accuracy / F1 | 15-20 своїх речень + правильна мітка (positive/negative) | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy), [`f1`](https://huggingface.co/spaces/evaluate-metric/f1) |
| zero-shot-classification | Accuracy | те саме + фіксований список `candidate_labels` | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy) |
| NER | F1 (entity-level) | кілька речень + список правильних сутностей (текст + тип) | [`seqeval`](https://huggingface.co/spaces/evaluate-metric/seqeval) |

Усі метрики рахуються через бібліотеку `evaluate` (той самий автор, що й `transformers`/`datasets`):
```python
import evaluate

metric = evaluate.load("rouge")  # або "accuracy", "f1", "seqeval", "squad", "sacrebleu"
metric.compute(predictions=[...], references=[...])
```
Формат `predictions`/`references` трохи відрізняється між метриками — дивіться приклад використання на сторінці кожної метрики (посилання в таблиці) або [HF Evaluate — Quick tour](https://huggingface.co/docs/evaluate/a_quick_tour).

### 5.2. Порівняйте моделі

На обраній задачі порівняйте щонайменше 2 різні моделі на однакових вхідних прикладах — не лише значення метрики, а й швидкість інференсу та розмір моделі (кількість параметрів).

### 5.3. Поекспериментуйте з налаштуваннями

Візьміть модель, що показала кращий результат, і поекспериментуйте з її налаштуваннями (temperature/top-p/top-k для генеративних задач, поріг впевненості для класифікаційних задач тощо) — щонайменше 3 зміни, по одній за раз, щоб зрозуміти, що саме впливає на метрику. Зведіть усі спроби (модель × налаштування → значення метрики) в невелику таблицю.

Коротко письмово: яка модель і яка комбінація налаштувань дала найкращий результат за обраною метрикою, і наскільки суттєво налаштування вплинули порівняно з вибором моделі?

### 5.1. Обрання задачі й метрики

In [17]:
!pip install evaluate

  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 2.4.0
    Uninstalling huggingface_hub-2.4.0:
      Successfully uninstalled huggingface_hub-2.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [evaluate]


In [18]:
!pip install scikit-learn

In [27]:
import evaluate
import time
import pandas as pd
from transformers import pipeline

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

### 5.2. Порівняння моделей

In [28]:
texts = [
    "Я дуже щасливий сьогодні!",                       #+
    "Цей фільм був неймовірно нудним.",                #-
    "Дякую за допомогу, ви врятували мій день.",       #+
    "Я розчарований результатом іспиту.",              #-
    "Погода чудова, хочеться прогулятися.",            #+
    "Сервіс жахливий, більше ніколи не прийду.",       #-
    "Мені сподобався цей ресторан.",                   #+
    "Я втомився від постійних проблем.",               #-
    "Ця книга надихає на великі звершення.",           #+
    "Все пішло не за планом."                          #-
]
references = ["positive", "negative", "positive", "negative", "positive",
              "negative", "positive", "negative", "positive", "negative"]

model1_name = "tabularisai/multilingual-sentiment-analysis"
model2_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

classifier1 = pipeline("sentiment-analysis", model=model1_name)
classifier2 = pipeline("sentiment-analysis", model=model2_name)

preds1_raw = classifier1(texts)
preds2_raw = classifier2(texts)

preds1 = [p['label'].lower() for p in preds1_raw]
preds2 = [p['label'].lower() for p in preds2_raw]

start_t1 = time.time()
time1 = time.time() - start_t1
params1 = sum(p.numel() for p in classifier1.model.parameters())

start_t2 = time.time()
time2 = time.time() - start_t2
params2 = sum(p.numel() for p in classifier2.model.parameters())

def normalize_label(label):
    label = label.lower()

    if "negative" in label or label in ["label_0", "label_1"]:
        return "negative"
    elif "positive" in label or label == "label_2":
        return "positive"
    else:
        return "neutral"

preds1 = [normalize_label(x) for x in preds1]
preds2 = [normalize_label(x) for x in preds2]

label_map = {"negative": 0, "positive": 1, "neutral": 2}

references_num = [label_map[x] for x in references]
preds1_num = [label_map[x] for x in preds1]
preds2_num = [label_map[x] for x in preds2]

acc1 = accuracy_metric.compute(
    predictions=preds1_num,
    references=references_num
)['accuracy']

f1_1 = f1_metric.compute(
    predictions=preds1_num,
    references=references_num,
    average='macro'
)['f1']

acc2 = accuracy_metric.compute(
    predictions=preds2_num,
    references=references_num
)['accuracy']

f1_2 = f1_metric.compute(
    predictions=preds2_num,
    references=references_num,
    average='macro'
)['f1']

print(f"Модель 1 ({model1_name}):")
print(f"  Параметрів: {params1/1e6:.1f} млн")
print(f"  Час інференсу: {time1:.4f} сек")
print(f"  Accuracy: {acc1:.2f}, F1: {f1_1:.2f}\n")

print(f"Модель 2 ({model2_name}):")
print(f"  Параметрів: {params2/1e6:.1f} млн")
print(f"  Час інференсу: {time2:.4f} сек")
print(f"  Accuracy: {acc2:.2f}, F1: {f1_2:.2f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Модель 1 (tabularisai/multilingual-sentiment-analysis):
  Параметрів: 135.3 млн
  Час інференсу: 0.0000 сек
  Accuracy: 0.80, F1: 0.80

Модель 2 (cardiffnlp/twitter-xlm-roberta-base-sentiment):
  Параметрів: 278.0 млн
  Час інференсу: 0.0000 сек
  Accuracy: 1.00, F1: 1.00


### 5.3. Експерименти з налаштуваннями

In [33]:
best_model_name = model2_name
best_classifier = classifier2

print(f"Експерименти з моделлю: {best_model_name}\n")

refs_num = [label_map[x] for x in references]

results = []

#--- БАЗОВИЙ РЕЗУЛЬТАТ (без змін) ---
preds_base = [normalize_label(p['label']) for p in best_classifier(texts)]
preds_base_num = [label_map[x] for x in preds_base]
acc_base = accuracy_metric.compute(predictions=preds_base_num, references=refs_num)['accuracy']
f1_base = f1_metric.compute(predictions=preds_base_num, references=refs_num, average='macro')['f1']
results.append({"Експеримент": "Базовий (без змін)", "Accuracy": acc_base, "F1 (macro)": f1_base})

#--- ЗМІНА 1: Поріг впевненості 0.8 ---
preds_thresh = []
for p in best_classifier(texts):
    if p['score'] < 0.8:
        preds_thresh.append("neutral")
    else:
        preds_thresh.append(normalize_label(p['label']))
preds_thresh_num = [label_map.get(x, 2) for x in preds_thresh] #2 - neutral
acc_t = accuracy_metric.compute(predictions=preds_thresh_num, references=refs_num)['accuracy']
f1_t = f1_metric.compute(predictions=preds_thresh_num, references=refs_num, average='macro')['f1']
results.append({"Експеримент": "Поріг впевненості 0.8", "Accuracy": acc_t, "F1 (macro)": f1_t})

#--- ЗМІНА 2: Об'єднання Neutral у Positive ---
preds_merge = []
for p in best_classifier(texts):
    lbl = normalize_label(p['label'])
    if lbl == "neutral":
        lbl = "positive" #Примусово
    preds_merge.append(lbl)
preds_merge_num = [label_map[x] for x in preds_merge]
acc_m = accuracy_metric.compute(predictions=preds_merge_num, references=refs_num)['accuracy']
f1_m = f1_metric.compute(predictions=preds_merge_num, references=refs_num, average='macro')['f1']
results.append({"Експеримент": "Об'єднання Neutral->Positive", "Accuracy": acc_m, "F1 (macro)": f1_m})

#--- ЗМІНА 3: Обрізка тексту (тільки перші 3 слова) ---
texts_truncated = [" ".join(t.split()[:3]) for t in texts]
preds_trunc_raw = best_classifier(texts_truncated)
preds_trunc = [normalize_label(p['label']) for p in preds_trunc_raw]
preds_trunc_num = [label_map[x] for x in preds_trunc]
acc_tr = accuracy_metric.compute(predictions=preds_trunc_num, references=refs_num)['accuracy']
f1_tr = f1_metric.compute(predictions=preds_trunc_num, references=refs_num, average='macro')['f1']
results.append({"Експеримент": "Обрізка до 3 слів", "Accuracy": acc_tr, "F1 (macro)": f1_tr})

df_final = pd.DataFrame(results)
df_final["Accuracy"] = df_final["Accuracy"].round(2)
df_final["F1 (macro)"] = df_final["F1 (macro)"].round(2)
display(df_final)

Експерименти з моделлю: cardiffnlp/twitter-xlm-roberta-base-sentiment



,Експеримент,Accuracy,F1 (macro)
0,Базовий (без змін),1.0,1.00
1,Поріг впевненості 0.8,0.7,0.55
2,Об'єднання Neutral->Positive,1.0,1.00
3,Обрізка до 3 слів,0.9,0.90


In [35]:
import pandas as pd
import os

BASE_DIR = "/content/drive/MyDrive/lab3"
os.makedirs(BASE_DIR, exist_ok=True)

gpt_data = [
    {
        "Категорія": "mini-GPT",
        "Модель/Варіант": "minigpt_v1_small",
        "Параметри": "~2.9M",
        "Контекст": 64,
        "Кількість шарів": 4,
        "Розмірність (embed)": 128,
        "val_loss": 6.149,
        "Accuracy": "-",
        "F1 (macro)": "-"
    },
    {
        "Категорія": "mini-GPT",
        "Модель/Варіант": "minigpt_v2_medium",
        "Параметри": "~8,9M",
        "Контекст": 128,
        "Кількість шарів": 6,
        "Розмірність (embed)": 256,
        "val_loss": 6.590,
        "Accuracy": "-",
        "F1 (macro)": "-"
    }
]

models_data = [
    {
        "Категорія": "NLP Pipeline (Sentiment)",
        "Модель/Варіант": model1_name,
        "Параметри": "~278M",
        "Контекст": "-",
        "Кількість шарів": "-",
        "Розмірність (embed)": "-",
        "val_loss": "-",
        "Accuracy": round(acc1, 2),
        "F1 (macro)": round(f1_1, 2)
    },
    {
        "Категорія": "NLP Pipeline (Sentiment)",
        "Модель/Варіант": model2_name,
        "Параметри": "~278M",
        "Контекст": "-",
        "Кількість шарів": "-",
        "Розмірність (embed)": "-",
        "val_loss": "-",
        "Accuracy": round(acc2, 2),
        "F1 (macro)": round(f1_2, 2)
    }
]

all_results = gpt_data + models_data
df_results = pd.DataFrame(all_results)

csv_path = os.path.join(BASE_DIR, "results.csv")
df_results.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Файл успішно збережено за шляхом: {csv_path}")
print("\nВміст файлу results.csv:")
display(df_results)

Файл успішно збережено за шляхом: /content/drive/MyDrive/lab3/results.csv

Вміст файлу results.csv:


,Категорія,Модель/Варіант,Параметри,Контекст,Кількість шарів,Розмірність (embed),val_loss,Accuracy,F1 (macro)
0,mini-GPT,minigpt_v1_small,~2.9M,64,4,128,6.149,-,-
1,mini-GPT,minigpt_v2_medium,"~8,9M",128,6,256,6.59,-,-
2,NLP Pipeline (Sentiment),tabularisai/multilingual-sentiment-analysis,~278M,-,-,-,-,0.8,0.8
3,NLP Pipeline (Sentiment),cardiffnlp/twitter-xlm-roberta-base-sentiment,~278M,-,-,-,-,1.0,1.0


Під час проведення експериментів з нашою моделлю ми отримали такі результати:

**Експеримент1:**
Accuracy впав до 0.7, F1 впав до 0.55. Це сталося тому, що модель впевнена у своїх відповідях, а штучне відсікання невпевнених прогнозів призвело до появи хибних "нейтральних" класів, яких немає у правильних відповідях.

**Експеримент2:**
Accuracy = 1.0, F1 = 1.0. Оскільки модель і так не видала жодного нейтрального прогнозу на цих реченнях, це налаштування не вплинуло на результат.

**Експеримент3:**
Accuracy впав до 0.9, F1 впав до 0.90. Це демонструє, що втрата контексту (коли речення обірване) негативно впливає на здатність моделі правильно визначати тональність.

> **Висновок:** Найкращий результат за обраною метрикою (Accuracy = 1.0) дала модель без жодних налаштувань (базовий варіант). Експерименти показали, що налаштування (поріг впевненості та обрізка тексту) можуть суттєво погіршити метрику, навіть якщо сама модель обрана вдало. Отже, вибір правильної моделі є важливішим фактором, ніж подальше "тюнінгування" її вихідних даних, а штучне втручання в роботу впевненої моделі може нашкодити.



---
## 6. Творчий застосунок

Побудуйте невеликий Gradio-застосунок на основі будь-якої готової моделі з розділів 4-5 — тема й формат повністю на ваш вибір, заохочується творчість і щось не зовсім очевидне (не просто "ще один чат-бот"). Це може бути й генеративна LLM через `transformers` (`AutoModelForCausalLM`/`AutoTokenizer` або `pipeline("text-generation", ...)`, той самий патерн, що й у демо 4.1-4.2 на Qwen) — `text-generation` уже входить до обов'язкових 8 задач розділу 4, тож це не вихід за межі завдання. Головне — щоб застосунок реально запускався і демонстрував конкретну, зрозумілу користувацьку цінність.

In [36]:
!pip install gradio

In [37]:
import gradio as gr
from transformers import pipeline
import torch

classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=0 if torch.cuda.is_available() else -1
)

def analyze_sentiment(text):
    if not text.strip():
        return "Будь ласка, введіть текст для аналізу.", ""

    result = classifier(text)[0]

    label_map = {
        "LABEL_0": "😡 Негативний (Negative)",
        "LABEL_1": "😐 Нейтральний (Neutral)",
        "LABEL_2": "😊 Позитивний (Positive)",
        "negative": "😡 Негативний (Negative)",
        "neutral": "😐 Нейтральний (Neutral)",
        "positive": "😊 Позитивний (Positive)"
    }

    raw_label = result['label']
    score = result['score']

    display_label = label_map.get(raw_label, raw_label)
    confidence = f"{score * 100:.2f}%"

    return display_label, confidence

with gr.Blocks(title="Аналізатор настрою", theme="soft") as demo:
    gr.Markdown("# 📝 Аналізатор настрою тексту")
    gr.Markdown("Введіть будь-який український текст (відгук, коментар, рецензію), і модель визначить його емоційне забарвлення.")

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(
                label="Ваш текст",
                placeholder="Наприклад: Цей сервіс просто жахливий, я більше ніколи не повернуся...",
                lines=4
            )
            submit_btn = gr.Button("Аналізувати", variant="primary")

        with gr.Column():
            output_label = gr.Textbox(label="Результат (Настрій)", interactive=False)
            output_score = gr.Textbox(label="Впевненість моделі", interactive=False)

    submit_btn.click(
        fn=analyze_sentiment,
        inputs=text_input,
        outputs=[output_label, output_score]
    )

    gr.Examples(
        examples=[
            ["Я дуже задоволений покупкою, якість супер!"],
            ["Товар прийшов зіпсований, гроші не повернули."],
            ["Ну, нормально, нічого особливого."]
        ],
        inputs=text_input
    )

demo.launch(debug=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_4374/1575959344.py:34: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Аналізатор настрою", theme="soft") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4792e114cd361c5e35.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4792e114cd361c5e35.gradio.live


---
### Як здавати лабу

У [форму](https://forms.gle/Z1gxmsDs656jqsDGA) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
3. **Файли всіх натренованих варіантів mini-GPT** (3.2) — кожен окремим `.pt` файлом.
4. **Логи TensorBoard** — папка `runs/` з усіма прогонами (і mini-GPT, і, за бажання, логи порівняння в розділі 5).
5. **Таблицю результатів** — `results.csv` з порівнянням варіантів mini-GPT (val_loss) і моделей з розділу 5.
6. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab3/transformers_demo.ipynb
/lab3/transformers_task.ipynb
/lab3/models/minigpt_v1.pt
/lab3/models/minigpt_v2.pt
/lab3/models/...        (усі інші варіанти з 3.2)
/lab3/runs/*
/lab3/results.csv
```